# DQN - Deep Q-Network

In the previous notebooks (Q-learning, SARSA), $Q(s,a)$ was always represented by a **table**: a two-entry array (state, action), with one cell per state. That works because dice-jack only has a handful of possible states (the score, from 0 to 22): 23 rows are enough.

In this notebook, we will train a pendulum to move to and stay balanced in a vertical position by acting on the torque of a joint. This pendulum is called **Acrobot**.

<img src="https://gymnasium.farama.org/_images/acrobot.gif" width="200">


In this environment, the state of the Acrobot is a vector of **6 real numbers** (cosine/sine of two angles, two angular velocities). Even discretizing each value into just 10 bins, a table covering every state would have $10^6$ = 1 million rows — and that coarse discretization would already throw away a huge amount of precision (two very close angles falling into two different bins would be treated as two completely unrelated states). With a truly continuous state, there are infinitely many possible states: an exhaustive table is simply impossible.

The idea behind **DQN (Deep Q-Network)**, introduced by DeepMind in 2015, is to replace the table with a **neural network** $Q_\theta(s,a)$ that *approximates* the Q-values instead of storing them one by one. The network takes a state as input and directly predicts the Q-value of each action.

Good news: DQN is still, fundamentally, the Q-learning algorithm you already know. Same Bellman equation, same epsilon-greedy policy, same *off-policy* nature. What changes is *how* $Q$ is represented and updated — and two technical complications appear, specific to this neural-network representation (an exact table is never subject to them). Rather than handing you these two ingredients ready-made, we'll discover them by walking through the problem: we'll first build a "naive" DQN that uses neither, observe why it learns poorly, then add the two ingredients one at a time, measuring the actual effect of each as we go.

**Key points of this notebook:**
- Why a table becomes impractical as soon as the state is continuous, and how a neural network solves that problem
- A minimal refresher on training a network with PyTorch, and the direct link with the Q-learning update rule you already know
- An "empty-handed" demonstration: a DQN with no replay buffer and no target network, to see concretely why these two ingredients are necessary
- The role of the replay buffer (experience memory) and its measured effect on learning
- The role of the target network (stabilizing the learning target) and its measured effect
- A full implementation of a DQN agent with PyTorch, trained and evaluated on **Acrobot-v1** (Gymnasium)
- Bonus: reusing the same agent on **LunarLander-v3**, a richer environment

## Presenting the Acrobot-v1 environment

Acrobot is a two-link arm hanging from a pivot. Only one of the two joints can receive a motor torque; the goal is to swing the arm high enough that its tip goes above a given line, in as few timesteps as possible.

- **States**: continuous vector of 6 values (cosine/sine of the two joint angles + the two angular velocities)
- **Actions**: 3 discrete actions (apply a negative, zero, or positive torque on the actuated joint)
- **Reward**: -1 at every timestep until the goal is reached, 0 once the arm's tip goes above the line (episode ends). The episode is truncated after 500 steps if it hasn't succeeded.
- **Policy**: epsilon-greedy, just like in Q-learning/SARSA

This dense reward (-1 everywhere) gives a learning signal that's usable from the very first episodes, unlike environments with very sparse rewards.

## 1. Baseline: a random agent

Before training anything, we need a point of comparison: how much reward does an agent get if it just picks actions at random? Same reflex as in the previous notebooks: you can only judge an agent's progress relative to a reference point.

**Exercise 1.1:**
Load the `Acrobot-v1` environment with `gymnasium`. Print its `observation_space` and `action_space`. Run an agent that picks random actions over a few dozen episodes, and compute the average reward obtained: this will be our baseline for judging DQN's progress.

In [ ]:
import random
from collections import deque
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
env = gym.make("Acrobot-v1")

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)


def random_baseline(env, num_episodes=20):
    """
    Runs a random policy on `env` and returns the total reward of each episode.

    Args:
        env: a Gymnasium environment exposing reset() / step() / action_space.sample().
        num_episodes (int): number of episodes to run.

    Returns:
        list[float]: the total reward obtained in each of the `num_episodes` episodes.
    """
    pass

In [ ]:
# TODO: call random_baseline(env) to get `baseline_rewards`, then print the average
# reward over these episodes. Keep this number around: it is the reference point
# you'll compare every DQN version to throughout the notebook.

### Bonus: watching the agent play

Reward curves are useful but a bit abstract. Since Acrobot is easy to visualize, we'll also **animate** an episode from time to time, to see concretely what the agent has (or hasn't) learned. The code below is provided directly (not an exercise): it isn't the point of this notebook, just a nice extra.

The idea: `gymnasium` can render each step as an image (`render_mode="rgb_array"`); we just need to collect these images along an episode, then stitch them together with `matplotlib.animation` to get a small video we can display inline in the notebook.

In [ ]:
from matplotlib import animation
from IPython.display import HTML


def record_episode(env_name, agent=None, max_steps=300):
    """Plays one episode and returns the list of rendered frames.
    If `agent` is None, plays a random policy; otherwise plays the agent greedily (epsilon=0)."""
    env = gym.make(env_name, render_mode="rgb_array")
    state, _ = env.reset()

    if agent is not None:
        original_epsilon = agent.epsilon
        agent.epsilon = 0.0

    frames = [env.render()]
    done = False
    steps = 0
    while not done and steps < max_steps:
        action = env.action_space.sample() if agent is None else agent.act(state)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        frames.append(env.render())
        steps += 1

    env.close()
    if agent is not None:
        agent.epsilon = original_epsilon
    return frames


def show_animation(frames, title="", skip=2, interval=40):
    """Displays a list of frames as an animation in the notebook (one frame out of every `skip`, to keep it light)."""
    frames = frames[::skip]
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.set_title(title)
    ax.axis("off")
    img = ax.imshow(frames[0])

    def update(i):
        img.set_data(frames[i])
        return [img]

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


show_animation(record_episode("Acrobot-v1"), title="Random policy")

## 2. A quick detour: how does a neural network learn?

Before diving into DQN, let's take a moment to recall how a neural network is trained with PyTorch. If you're already comfortable with `loss.backward()` and `optimizer.step()`, feel free to jump straight to section 3.

A neural network is a function $f_\theta(x)$ whose parameters $\theta$ (the weights) are adjusted so that $f_\theta(x)$ gets closer to some target value $y$. At each iteration:

1. **Forward pass**: compute the prediction $f_\theta(x)$.
2. **Loss**: measure the error between the prediction and the target, for example with the squared error $\mathcal{L} = (y - f_\theta(x))^2$.
3. **Backward pass**: PyTorch automatically computes the gradient of the loss with respect to *every* parameter ($\partial \mathcal{L}/\partial \theta$), via `loss.backward()`.
4. **Optimizer step**: update each parameter in the direction that reduces the loss, via `optimizer.step()`. For the simplest version of this update (plain gradient descent): $\theta \leftarrow \theta - \text{lr} \cdot \partial \mathcal{L}/\partial \theta$.

**Does this ring a bell?** Look at the tabular Q-learning update rule you've already implemented:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \big( r + \gamma \max_{a'} Q(s',a') - Q(s,a) \big)$$

This is *exactly* one gradient descent step on the loss $\mathcal{L} = \big(r + \gamma \max_{a'} Q(s',a') - Q(s,a)\big)^2$, where the only adjustable parameter is the cell $Q(s,a)$ itself, and $\alpha$ plays the role of the learning rate! Replacing the table with a network therefore doesn't change the underlying principle at all: we're still adjusting parameters to reduce the gap between a prediction and a Bellman target. The difference is that instead of a single independent parameter per cell, we now have thousands of them, shared across all states — which makes computing the gradient by hand impractical, hence `loss.backward()`.

**Exercise 2.1:**
To get familiar with this mechanism before applying it to DQN, train a tiny network (a single `nn.Linear(1, 1)` layer is enough) to recover the relationship $y = 2x + 1$ from noisy data. Generate synthetic data, write the training loop (forward, MSE loss, `zero_grad`, `backward`, `step`), plot the loss over iterations, and check that the learned weights get close to 2 and 1.

In [ ]:
# TODO:
# 1. Generate synthetic data: x_train (e.g. torch.linspace) and
#    y_train = 2 * x_train + 1 + some noise (e.g. torch.randn_like).
# 2. Create the model: a single nn.Linear(1, 1) layer.
# 3. Create an optimizer (e.g. optim.Adam) on the model's parameters.
# 4. Training loop, for a couple hundred iterations:
#      - forward pass: y_pred = model(x_train)
#      - loss = nn.functional.mse_loss(y_pred, y_train)
#      - optimizer.zero_grad() ; loss.backward() ; optimizer.step()
#      - store loss.item() in a list to plot afterwards
# 5. Print the learned weight and bias (model.weight.item(), model.bias.item())
#    and check they are close to 2 and 1.
# 6. Plot the loss curve, and a scatter of the data with the fitted line on top.

## 3. From table to function: the Q-network

In tabular form, `Q` was an array: `Q[s][a]` directly returned a number stored in memory. With a network, $Q_\theta(s)$ is a **computation**: we feed in a state, and the network returns a vector of Q-values, one per action — as if we were reading an entire row of the table at once, except that row doesn't exist explicitly in memory: it is recomputed on the fly from the network's weights.

What **doesn't change** compared to tabular Q-learning:
- The Bellman equation: $Q(s,a) \leftarrow Q(s,a) + \alpha \big( r + \gamma \max_{a'} Q(s',a') - Q(s,a) \big)$
- The *off-policy* nature: the target uses $\max_{a'} Q(s',a')$, not the action actually chosen
- The epsilon-greedy policy for balancing exploration/exploitation

What **changes**:
- $Q(s,a)$ is no longer read from/written to a table, but predicted by a neural network $Q_\theta(s,a)$ whose weights $\theta$ are adjusted via gradient descent (exactly like in the small exercise above)
- The cell-by-cell update becomes a minimization of the loss $\big(r + \gamma \max_{a'} Q_\theta(s',a') - Q_\theta(s,a)\big)^2$

The network takes a state as input (a vector of `state_dim` real values) and outputs one $Q(s,a)$ value **per possible action** (a vector of `action_dim` values). A simple architecture (2 hidden layers) is more than enough for Acrobot.

**Exercise 3.1:**
Complete the `QNetwork` class: the network should take `state_dim` and `action_dim` in its constructor, and define a small MLP (multi-layer perceptron) that maps one to the other.

**Exercise 3.2:**
Before any training, instantiate a `QNetwork` for Acrobot, feed it a real state from the environment (e.g. the state returned by `env.reset()`), and print the predicted Q-values. They'll obviously be meaningless (the network has random, untrained weights) — but check that the shape is correct: a vector of length `action_dim`. This is the same kind of sanity check as verifying that a freshly zero-initialized Q-table has the right dimensions.

In [ ]:
class QNetwork(nn.Module):
    """Approximates Q(s,a): input = state, output = one Q-value per action."""

    def __init__(self, state_dim, action_dim, hidden_dim=128):
        """
        Args:
            state_dim (int): dimensionality of the state vector.
            action_dim (int): number of discrete actions.
            hidden_dim (int): size of the hidden layers.

        Attributes to set up:
            self.net: a small MLP going from state_dim to action_dim, e.g.
                Linear(state_dim, hidden_dim) -> ReLU
                -> Linear(hidden_dim, hidden_dim) -> ReLU
                -> Linear(hidden_dim, action_dim)
        """
        super().__init__()
        pass

    def forward(self, state):
        """
        Args:
            state (torch.Tensor): shape (batch_size, state_dim).

        Returns:
            torch.Tensor: predicted Q-values, shape (batch_size, action_dim).
        """
        pass

In [ ]:
# TODO:
# 1. Create env = gym.make("Acrobot-v1"), and read state_dim / action_dim from it.
# 2. Instantiate a QNetwork(state_dim, action_dim) (moved to `device`).
# 3. Get a real state from env.reset(), convert it to a float32 tensor with a
#    batch dimension (unsqueeze(0)), and pass it through the network inside a
#    torch.no_grad() block.
# 4. Print the state and the predicted Q-values, and check the output has
#    length action_dim (it will be random nonsense: the network isn't trained yet).

## 4. A first DQN agent, with no safety net

We now have a way to predict $Q(s,a)$ with a network. Let's try the most direct approach: reuse exactly the tabular Q-learning loop (pick an epsilon-greedy action, step, compute the Bellman target, update), but replace the table-cell update with a gradient step on the network — **immediately after each transition**, without storing anything, and using the same network for both the prediction and the target.

**Exercise 4.1:**
Complete the `NaiveDQNAgent` class:
- `act(state)`: epsilon-greedy policy (identical to what you already did in Q-learning/SARSA)
- `learn_step(state, action, reward, next_state, done)`: computes $Q_\theta(s,a)$ (forward pass), the Bellman target $r + \gamma \max_{a'} Q_\theta(s',a')$ **with this very same network** (no separate copy), the MSE loss between the two, then one gradient step
- `decay_epsilon()`: identical to the previous notebooks

**Exercise 4.2:**
Train this agent on Acrobot for 150 episodes (same loop structure as in section 1, but calling `learn_step` at every step) and plot the reward curve. Compare it to the random baseline. Keep this curve handy, we'll compare it to the following versions.

In [ ]:
class NaiveDQNAgent:
    """'Empty-handed' DQN: no replay buffer, no target network. Learns transition by transition,
    using the same network for the prediction and for the target."""

    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon=1.0, device=device):
        """
        Args:
            state_dim (int): dimensionality of the state vector.
            action_dim (int): number of discrete actions.
            lr (float): learning rate for the optimizer.
            gamma (float): discount factor.
            epsilon (float): initial exploration rate.
            device: torch device the network should live on.

        Attributes to set up:
            self.action_dim, self.gamma, self.epsilon, self.device: store the arguments.
            self.q_network: a QNetwork(state_dim, action_dim) moved to `device`.
            self.optimizer: an Adam optimizer over self.q_network's parameters, with learning rate `lr`.
        """
        pass

    def act(self, state):
        """
        Epsilon-greedy action selection.

        Args:
            state (np.ndarray): the current state.

        Returns:
            int: the chosen action index (random with probability epsilon,
                 otherwise argmax of self.q_network's predicted Q-values).
        """
        pass

    def learn_step(self, state, action, reward, next_state, done):
        """
        Performs one online gradient step from a single transition.

        Args:
            state (np.ndarray): the state before the action.
            action (int): the action taken.
            reward (float): the reward received.
            next_state (np.ndarray): the state after the action.
            done (float or bool): whether `next_state` is terminal.

        Returns:
            float: the loss value for this step.

        Steps:
            1. Compute Q(s, a) with self.q_network (forward pass on `state`,
               then pick out the value for `action`).
            2. Compute the Bellman target r + gamma * max_a' Q(s', a') using
               THIS SAME network (no target network here), inside a
               torch.no_grad() block. Remember to zero out the future term
               when `done` is true.
            3. Compute the MSE loss between the two.
            4. optimizer.zero_grad() ; loss.backward() ; optimizer.step().
        """
        pass

    def decay_epsilon(self, min_epsilon=0.01, decay_rate=0.995):
        """
        Decays epsilon geometrically (epsilon *= decay_rate), never going below `min_epsilon`.
        """
        pass

In [ ]:
env = gym.make("Acrobot-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

naive_agent = NaiveDQNAgent(state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon=1.0)
num_ablation_episodes = 150

# TODO: train `naive_agent` on Acrobot for `num_ablation_episodes` episodes.
#
# For each episode:
#   - reset the environment
#   - loop until done: pick an action with naive_agent.act(state), step the
#     environment, immediately call naive_agent.learn_step(...) (no storage),
#     move to next_state
#   - after the episode, call naive_agent.decay_epsilon()
#   - append the episode's total reward to a list called `rewards_naive`
#     (reused later, keep this exact name)
#
# Then plot `rewards_naive` (raw + moving average) together with a horizontal
# line for the random baseline, the same way you plotted Q-learning/SARSA
# reward curves before.

In [ ]:
show_animation(record_episode("Acrobot-v1", agent=naive_agent), title="Naive DQN (150 episodes)")

### Why doesn't it learn (or barely)?

You should observe a curve that improves very little, if at all, and often stays close to the random baseline. Two causes, both specific to using a neural network (an exact table is never subject to them):

1. **Correlation between consecutive transitions.** Two consecutive transitions from the same episode look a lot alike (the state at $t+1$ resembles the state at $t$). Training the network on such correlated examples, in the order they arrive, is like feeding it very little variety at once — a bit like revising for an exam by only re-reading the last page of your notes over and over. The network overfits to the recent trajectory and "forgets" what it had just learned before.

2. **Moving target.** The Bellman target $r + \gamma \max_{a'} Q_\theta(s',a')$ is computed with the *same* network $Q_\theta$ that we're currently modifying. At every gradient step, we adjust $\theta$ to bring the prediction closer to the target — but the target itself depends on $\theta$, so it moves along with the prediction! It's like trying to catch your own shadow: every step you take toward the target moves it a bit further away. In tabular form this wasn't a problem, since each table cell was independent of the others; with a network, the weights are shared across all states, so correcting $Q_\theta(s,a)$ also slightly shifts $Q_\theta(s',a')$ for any other state — including the one used as the target.

The next two sections each add a remedy for one of these two problems, one at a time, so we can observe the effect of each separately before combining them.

## 5. First ingredient: the Replay Buffer

The **replay buffer** addresses the temporal correlation problem: instead of learning from the transition that just happened, we store it in a memory $(s, a, r, s', done)$, and train the network on **randomly sampled mini-batches** drawn from that memory. This breaks the temporal correlation (a batch mixes transitions from different moments and different episodes) and lets us reuse the same experience multiple times.

**Exercise 5.1:**
Complete the `ReplayBuffer` class with three methods: `push` (add a transition), `sample` (draw a random mini-batch of size `batch_size`), and `__len__`.

**Exercise 5.2:**
Take the logic of `NaiveDQNAgent` and adapt it to draw its mini-batches from a replay buffer instead of learning from the current transition alone. The Bellman target still uses **the same network** for now (no target network yet): we want to isolate the effect of the buffer alone before adding the second ingredient. Call this class `BufferOnlyAgent`, with the methods `act`, `store_transition`, `learn` (which draws a batch and takes a gradient step on it), and `decay_epsilon`. Train it for 150 episodes and compare the resulting curve to the naive DQN from section 4.

In [ ]:
class ReplayBuffer:
    """Memory of past transitions, allows random sampling for training."""

    def __init__(self, capacity=50_000):
        """
        Args:
            capacity (int): maximum number of transitions kept in memory.

        Attributes to set up:
            self.buffer: a collections.deque with maxlen=capacity.
        """
        pass

    def push(self, state, action, reward, next_state, done):
        """Appends one transition (state, action, reward, next_state, done) to the buffer."""
        pass

    def sample(self, batch_size):
        """
        Draws a random mini-batch of `batch_size` transitions.

        Returns:
            tuple of 5 np.ndarray (states, actions, rewards, next_states, dones),
            each of length `batch_size`, with dtypes float32 / int64 / float32 / float32 / float32.

        Hint: random.sample(self.buffer, batch_size) gives you a list of
        (state, action, reward, next_state, done) tuples; zip(*batch) transposes
        it into 5 separate sequences you can turn into np.array.
        """
        pass

    def __len__(self):
        """Returns the current number of transitions stored in the buffer."""
        pass

In [ ]:
class BufferOnlyAgent:
    """DQN with a replay buffer, but the target still uses the same network (no target network)."""

    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon=1.0,
                 buffer_capacity=50_000, batch_size=64, device=device):
        """
        Args:
            state_dim, action_dim, lr, gamma, epsilon, device: same meaning as in NaiveDQNAgent.
            buffer_capacity (int): capacity of the replay buffer.
            batch_size (int): mini-batch size drawn at each learning step.

        Attributes to set up:
            self.action_dim, self.gamma, self.epsilon, self.batch_size, self.device: store the arguments.
            self.q_network: a QNetwork(state_dim, action_dim) moved to `device`.
            self.optimizer: an Adam optimizer over self.q_network's parameters.
            self.replay_buffer: a ReplayBuffer(buffer_capacity).
        """
        pass

    def act(self, state):
        """Epsilon-greedy action selection (identical to NaiveDQNAgent.act)."""
        pass

    def store_transition(self, state, action, reward, next_state, done):
        """Pushes the transition into self.replay_buffer."""
        pass

    def learn(self):
        """
        Draws a mini-batch from the replay buffer and takes one gradient step.

        Returns:
            float or None: the loss value, or None if the buffer doesn't yet
            contain at least `self.batch_size` transitions.

        Steps:
            1. If len(self.replay_buffer) < self.batch_size, return None.
            2. Sample a batch, convert each array to a torch tensor.
            3. Compute Q(s,a) for the actions actually taken (self.q_network(states),
               then gather along the action dimension).
            4. Compute the Bellman target r + gamma * max_a' Q(s', a') using THIS
               SAME network (no target network here), inside torch.no_grad(),
               zeroing the future term where done=1.
            5. MSE loss between the two, then zero_grad / backward / step.
        """
        pass

    def decay_epsilon(self, min_epsilon=0.01, decay_rate=0.995):
        """Decays epsilon geometrically, never going below `min_epsilon`."""
        pass

In [ ]:
env = gym.make("Acrobot-v1")
buffer_agent = BufferOnlyAgent(state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon=1.0, batch_size=64)

# TODO: train `buffer_agent` on Acrobot for `num_ablation_episodes` episodes.
#
# Same loop as for the naive agent, except at every step you should call
# buffer_agent.store_transition(...) followed by buffer_agent.learn() (instead
# of a single learn_step call). Store the episode rewards in a list called
# `rewards_buffer_only`.
#
# Then plot `rewards_buffer_only` and `rewards_naive` (moving averages) on the
# same graph, together with the random baseline, to compare the two.

In [ ]:
show_animation(record_episode("Acrobot-v1", agent=buffer_agent), title="DQN + Replay buffer (150 episodes)")

You should observe a clear improvement over the naive DQN: the buffer decorrelates the data and lets each experience be reused multiple times, which speeds up and stabilizes learning. There is often still some instability left (occasional drops in performance), though, since the second problem — the moving target — hasn't been fixed yet. That's the subject of the next section.

## 6. Second ingredient: the Target Network

To stabilize the target, we introduce a **frozen copy** of the network, called the *target network* $Q_{\theta^-}$, used only to compute the Bellman target: $r + \gamma \max_{a'} Q_{\theta^-}(s',a')$. The main network $Q_\theta$ keeps being trained at every step as before, but the target it's chasing no longer moves at every gradient step: it stays fixed for several hundred steps, then gets updated all at once by copying the weights of $Q_\theta$ into $Q_{\theta^-}$.

To reuse the analogy from section 4: instead of chasing a shadow that constantly moves, we now aim at a target that stays still for a while, giving us time to get closer, before it gets repositioned.

**Exercise 6.1:**
Complete the `DQNAgent` class, which combines everything we've built so far:
- `__init__`: the main network, the target network (initialized with the same weights), the optimizer, the replay buffer
- `act(state)`: epsilon-greedy policy (unchanged)
- `store_transition(...)`: adds a transition to the replay buffer (unchanged)
- `learn()`: draws a mini-batch, computes the target **with the target network**, computes the MSE loss against the main network's prediction, takes a gradient descent step
- `update_target_network()`: copies the main network's weights into the target network

**Exercise 6.2:**
Add a `decay_epsilon()` method (identical to the previous versions). Then train the full agent on Acrobot for 300 episodes, and plot on the same graph the three curves obtained in this notebook (naive, +buffer, +buffer+target network) over the first 150 episodes of each, to compare their learning speed at an equal episode budget.

In [ ]:
class DQNAgent:
    """Full DQN: replay buffer + target network."""

    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon=1.0,
                 buffer_capacity=50_000, batch_size=64, target_update_freq=500,
                 device=device):
        """
        Args:
            state_dim, action_dim, lr, gamma, epsilon, device: same meaning as before.
            buffer_capacity (int): capacity of the replay buffer.
            batch_size (int): mini-batch size drawn at each learning step.
            target_update_freq (int): number of learning steps between two
                copies of the main network's weights into the target network.

        Attributes to set up:
            self.action_dim, self.gamma, self.epsilon, self.batch_size,
                self.target_update_freq, self.device: store the arguments.
            self.q_network: a QNetwork(state_dim, action_dim) moved to `device`.
            self.target_network: another QNetwork(state_dim, action_dim), moved
                to `device`, initialized with the SAME weights as self.q_network
                (load_state_dict), and put in eval() mode.
            self.optimizer: an Adam optimizer over self.q_network's parameters.
            self.replay_buffer: a ReplayBuffer(buffer_capacity).
            self.learn_step_counter: an int counter starting at 0, used to know
                when to update the target network.
        """
        pass

    def act(self, state):
        """Epsilon-greedy action selection (identical to the previous agents)."""
        pass

    def store_transition(self, state, action, reward, next_state, done):
        """Pushes the transition into self.replay_buffer."""
        pass

    def learn(self):
        """
        Draws a mini-batch from the replay buffer and takes one gradient step.

        Returns:
            float or None: the loss value, or None if the buffer doesn't yet
            contain at least `self.batch_size` transitions.

        Steps:
            1. If len(self.replay_buffer) < self.batch_size, return None.
            2. Sample a batch, convert each array to a torch tensor.
            3. Compute Q(s,a) with self.q_network for the actions actually taken.
            4. Compute the Bellman target r + gamma * max_a' Q(s', a') using
               self.target_network (not self.q_network!), inside torch.no_grad(),
               zeroing the future term where done=1.
            5. MSE loss between the two, then zero_grad / backward / step.
            6. Increment self.learn_step_counter; every `target_update_freq`
               steps, call self.update_target_network().
        """
        pass

    def update_target_network(self):
        """Copies self.q_network's weights into self.target_network."""
        pass

    def decay_epsilon(self, min_epsilon=0.01, decay_rate=0.995):
        """Decays epsilon geometrically, never going below `min_epsilon`."""
        pass

In [ ]:
env = gym.make("Acrobot-v1")

agent = DQNAgent(state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon=1.0,
                 batch_size=64, target_update_freq=500)
num_episodes = 300

# TODO: train `agent` on Acrobot for `num_episodes` episodes (same loop as for
# `buffer_agent`: act, step, store_transition, learn, and decay_epsilon after
# each episode). Store the episode rewards in `rewards_history` and the
# epsilon value after each episode in `epsilons_history`.

In [ ]:
# TODO: on a single graph, plot the moving-average reward curves of the three
# agents over their first `num_ablation_episodes` episodes:
#   - rewards_naive            ("Naive (nothing)")
#   - rewards_buffer_only      ("+ Replay buffer")
#   - rewards_history[:num_ablation_episodes]   ("+ Replay buffer + Target network")
# Add the random baseline as a horizontal reference line.

On this graph, the full version (replay buffer + target network) should learn noticeably faster than the other two, at an equal number of episodes. This is the empirical demonstration of what we anticipated in section 4: both ingredients bring a real gain, and combine well together.

Let's now continue with the full agent trained over 300 episodes, and look at its complete curve as well as the decay of its epsilon.

In [ ]:
window = 20
moving_avg = np.convolve(rewards_history, np.ones(window) / window, mode="valid")

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(rewards_history, alpha=0.3, label="Reward per episode", color="lightblue")
plt.plot(range(window - 1, len(rewards_history)), moving_avg, label="Moving average", color="red", linewidth=2)
plt.axhline(np.mean(baseline_rewards), color="gray", linestyle="--", label="Random baseline")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("Performance of the full DQN on Acrobot-v1")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epsilons_history, color="green")
plt.xlabel("Episode")
plt.ylabel("Epsilon")
plt.title("Epsilon decay")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluation

**Exercise 7.1:**
Test the trained agent under a purely greedy policy (epsilon=0, like in the `test_agent` function from the Q-learning/SARSA notebook) over several episodes, and compute the average reward obtained.

In [ ]:
def evaluate_agent(agent, env, num_episodes=20):
    """
    Evaluates the agent under a pure greedy policy (no exploration).

    Args:
        agent: any of the agents defined in this notebook (must have an
            `epsilon` attribute and an `act(state)` method).
        env: the Gymnasium environment to evaluate on.
        num_episodes (int): number of evaluation episodes.

    Returns:
        list[float]: the total reward obtained in each evaluation episode.

    Steps:
        1. Save agent.epsilon, then temporarily set it to 0.0.
        2. Run `num_episodes` episodes acting greedily, collecting the total
           reward of each.
        3. Restore agent.epsilon to its original value before returning.
    """
    pass

In [ ]:
# TODO: call evaluate_agent(agent, env) to get `eval_rewards`, then print the
# average and median reward, compared to the random baseline.
#
# Finally, save the trained network's weights with torch.save, e.g.:
#   timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#   torch.save(agent.q_network.state_dict(), f"dqn_acrobot_{timestamp}.pt")

And finally, to end on a high note, here's the full agent (replay buffer + target network) in action:

In [ ]:
show_animation(record_episode("Acrobot-v1", agent=agent), title="Full DQN (300 episodes)")

## Conclusion

DQN remains, in its core principle, the Q-learning algorithm you already know: same Bellman equation, same **off-policy** nature (the target uses $\max_{a'}$, not the action actually chosen next), same epsilon-greedy policy. The exact table becomes a neural network, which introduces two stability problems that you observed empirically in this notebook:

- **With nothing** (section 4): the network learns from correlated transitions and chases a target that moves along with it → essentially no learning.
- **+ Replay buffer** (section 5): decorrelating the data via randomly sampled mini-batches noticeably speeds up learning, but the moving target still creates instability.
- **+ Target network** (section 6): freezing the target for several hundred steps stabilizes training and enables much faster, steadier convergence.

| | Tabular Q-learning / SARSA | DQN |
|---|---|---|
| Representation of Q | exact table | neural network |
| Update | in-place, transition by transition | gradient descent on a mini-batch |
| Stability | a non-issue | requires replay buffer + target network |
| Derived policy | argmax over the table | argmax over the network's output |
| On/off-policy | Q-learning = off-policy, SARSA = on-policy | off-policy (like Q-learning) |
| State space | discrete, small | continuous or very large |

To go further, try now reusing exactly the same classes on a richer environment: **LunarLander-v3**.

## To go further: LunarLander-v3

LunarLander-v3 is a richer environment: piloting a lunar module (8 state values, 4 discrete actions) to land it smoothly between two flags. No new class is needed: `QNetwork`, `ReplayBuffer`, and `DQNAgent` (the full version from section 6) are directly reusable, only `state_dim` and `action_dim` change.

Installation required before running the next cell:
```
pip install "gymnasium[box2d]"
```

Note: LunarLander requires more episodes than Acrobot to converge (the environment has richer dynamics and reward shaping), so training will take longer. An average reward above 200 over 100 consecutive episodes is considered solved.

In [ ]:
env_lunar = gym.make("LunarLander-v3")
state_dim = env_lunar.observation_space.shape[0]
action_dim = env_lunar.action_space.n

agent_lunar = DQNAgent(state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon=1.0,
                       batch_size=64, target_update_freq=1000)
num_episodes = 500

# TODO: train `agent_lunar` on LunarLander-v3 for `num_episodes` episodes,
# exactly like you trained `agent` on Acrobot in section 6 (act, step,
# store_transition, learn, decay_epsilon). Store the episode rewards in
# `rewards_history_lunar`.
#
# Then plot the reward curve (raw + moving average), together with a
# horizontal line at 200 (the "solved" threshold).

In [ ]:
show_animation(record_episode("LunarLander-v3", agent=agent_lunar), title="DQN on LunarLander-v3")